In [4]:
# =============================================================================
# Project:  lv-syn-grid
# Author:   Roman S.
# Created:  2025
#
# Description:
# Preparation of photovoltaic (PV) systems for the integration into the generated 
# low-voltage network model, including the representation of distributed generation for
# simulation purposes.
#
# License: MIT License
#
# =============================================================================

import pvlib
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gp
import os

from pvlib import location, pvsystem, modelchain
from pvlib.location import Location
from pvlib.modelchain import ModelChain
from pvlib.pvsystem import PVSystem, FixedMount
from pvlib.temperature import TEMPERATURE_MODEL_PARAMETERS

from scripts import parameters as prm

plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']

In [ ]:
PV_START_DATE = '2025-06-21 00:00'
PV_END_DATE = '2025-06-21 23:45'

In [ ]:
b_gdf = gp.read_parquet(prm.FILEPATH_DATA + 'buildings.parquet')
pv_gdf = gp.read_file(prm.FILEPATH_DATA + 'pv_systems.csv')  # read the csv file and store it into a geopandas data frame

In [ ]:
pv_gdf = pv_gdf.rename(columns={
    "Name": "cardinal_direction",
    "Beschreibung": "nr"
})

direction_to_deg = {
    "N": 0,
    "NO": 45,
    "O": 90,
    "SO": 135,
    "S": 180,
    "SW": 225,
    "W": 270,
    "NW": 315
}

pv_gdf["orientation"] = (
    pv_gdf["cardinal_direction"].map(direction_to_deg)                 # mapping to degree
)

pv_gdf.crs = 'EPSG:4326'
pv_gdf = pv_gdf.to_crs('EPSG:9273')

pv_gdf['module_p_kw'] = 24.0
pv_gdf['inverter_p_kw'] = 24.0

In [ ]:
b_cent = b_gdf[["id", "centroid"]].copy()
b_cent = gp.GeoDataFrame(b_cent, geometry="centroid", crs='EPSG:9273')
b_cent

# find next building to each identified pv system
pv_matched = gp.sjoin_nearest(
    pv_gdf,
    b_cent[["id", "centroid"]],
    how="left",
    distance_col="dist"
).drop(columns=["index_right"])

pv_matched

In [ ]:
def create_pv_profile(module_p_kw, inverter_p_kw, module_gamma=-0.004, inverter_eta=0.96, surface_tilt=30, surface_azimuth=180) -> np.ndarray:
    module_parameters = {'pdc0': module_p_kw*1e3, 'gamma_pdc': module_gamma }
    inverter_parameters = { 'pdc0': inverter_p_kw*1e3,
                            'eta_inv_nom': inverter_eta }

    system = pvsystem.PVSystem( surface_tilt=surface_tilt,
                                surface_azimuth=surface_azimuth,
                                inverter_parameters=inverter_parameters,
                                module_parameters=module_parameters,
                                module_type='glass_polymer',
                                racking_model='open_rack')

    pdc = system.pvwatts_dc(g_poa_effective=1000, temp_cell=30)

    location = pvlib.location.Location(48.2082, 16.3738, 'Europe/Vienna')  # Vienna

    # weather data (clear sky)
    times = pd.date_range(start=PV_START_DATE, end=PV_END_DATE, freq='15min', tz=location.tz)
    weather = location.get_clearsky(times)

    mc = ModelChain(system,
                location,
                dc_model='pvwatts',
                ac_model='pvwatts',
                aoi_model='no_loss')

    # run the model
    mc.run_model(weather)

    return np.array(mc.results.ac)

In [ ]:
def apply_pv_profile(row):
    return create_pv_profile(
        module_p_kw=row["module_p_kw"],
        inverter_p_kw=row["inverter_p_kw"],
        module_gamma=-0.004,
        inverter_eta=0.96,
        surface_tilt=30,
        surface_azimuth=row["orientation"]
    )

pv_matched["pv_profile"] = pv_matched.apply(apply_pv_profile, axis=1)
pv_matched

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.dates as mdates


plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']

# color mapping
orientation_colors = {
    90:  "tab:orange",   # E
    135: "tab:green",    # SE
    180: "tab:red",      # S
    225: "tab:purple",   # SW
    270: "tab:blue"      # W
}

plt.figure(figsize=(10, 6))
time_index = pd.date_range(PV_START_DATE, periods=96, freq="15min")

for orientation, group in pv_matched.groupby("orientation"):
    mean_profile = group["pv_profile"].apply(pd.Series).mean()
    color = orientation_colors.get(orientation, "gray")
    plt.plot(time_index, mean_profile / 1e3, label=f"{orientation}"+r"$^{\circ}$", color=color, linewidth=2)


plt.legend(title=r"azimuth $\left[^{\circ} \right]$", title_fontsize=16, fontsize=16)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=2))

plt.grid(True)
plt.xlabel("Tageszeit", fontsize=18)
plt.ylabel("PV-Leistung [kW]", fontsize=18)
plt.xticks(fontsize=18, rotation=45)
plt.yticks(fontsize=18, rotation=0)
plt.tight_layout()
plt.xlim(pd.Timestamp(PV_START_DATE), pd.Timestamp(PV_END_DATE))
plt.tight_layout()
plt.savefig('pvmodel.svg', format='svg')
plt.show()

In [ ]:
pv_matched.to_parquet(prm.FILEPATH_DATA + prm.FILEPATH_PV_PARQUET)